In [ ]:
from pathlib import Path
import numpy as np
import tifffile as tiff

In [ ]:
from split_dataset import SplitDataset
from fimpy.pipeline.alignment import align_2p_volume
import json
import flammkuchen as fl
from glob import glob

import pandas as pd
from split_dataset import SplitDataset
from bouter import Experiment
from fimpy.pipeline.general import calc_f0, dff
from motions.utilities import stim_vel_dir_dataframe, quantize_directions
from scipy.interpolate import interp1d 
from scipy.signal import convolve2d
import colorspacious
import napari
import matplotlib.pyplot as plt

from fimpylab.core.twop_experiment import TwoPExperiment

In [ ]:
# make sensory regressors. requires old bouter stimulus_param_log.
def make_sensory_regressors(exp, n_dirs=8, upsampling=5, sampling=1/3):
    stim = stim_vel_dir_dataframe(exp)
    bin_centres, dir_bins = quantize_directions(stim.theta)
    ind_regs = np.zeros((n_dirs, len(stim)))
    for i_dir in range(n_dirs):
        ind_regs[i_dir, :] = (np.abs(dir_bins - i_dir) < 0.1) & (stim.vel > 0.1)  

    dt_upsampled = sampling / upsampling
    t_imaging_up = np.arange(0, stim.t.values[-1], dt_upsampled)
    reg_up = interp1d(stim.t.values, ind_regs, axis=1, fill_value="extrapolate")(
        t_imaging_up
    )
    
    # 6s kernel
    u_steps = t_imaging_up.shape[0]
    u_time = np.arange(u_steps) * dt_upsampled
    decay = np.exp(-u_time / (1.5 / np.log(2)))
    kernel = decay / np.sum(decay)
    
    convolved = convolve2d(reg_up, kernel[None, :])[:, 0:u_steps]
    reg_sensory = convolved[:, ::upsampling]

    return pd.DataFrame(reg_sensory.T, columns=[f"motion_{i}" for i in range(n_dirs)])

In [ ]:
# calculate directional tuning from dF/F traces, px-wise
def get_tuning_map(img, sens_regs, n_dirs=8):
    traces = img.reshape(img.shape[0], -1)

    n_t = sens_regs.shape[0]
    reg = sens_regs.values.T @ traces[:n_t, :]
    reg = reg.reshape(reg.shape[0], img.shape[-2], img.shape[-1])
    
    print(np.shape(img))
    print(np.shape(traces))
    # tuning vector
    bin_centers, bins = quantize_directions([0], n_dirs)
    vectors = np.stack([np.cos(bin_centers), np.sin(bin_centers)], 0)
    reg_vectors = np.reshape(
        vectors @ np.reshape(reg[:, :, :], (n_dirs, -1)),
        (2,) + reg.shape[1:],
    )
    
    
    print(np.shape(reg_vectors))
    
    angle = np.arctan2(reg_vectors[1], reg_vectors[0])
    amp = np.sqrt(np.sum(reg_vectors ** 2, 0))

    return amp, angle

In [ ]:
# make a color map

def JCh_to_RGB255(x):
    output = np.clip(colorspacious.cspace_convert(x, "JCh", "sRGB1"), 0, 1)
    return (output * 255).astype(np.uint8)

def color_stack(
        amp,
        angle,
        hueshift=2.5,
        amp_percentile=80,
        maxsat=50,
        lightness_min=100,
        lightness_delta=-40,
    ):
    output_lch = np.empty(amp.shape + (3,))
    maxamp = np.percentile(amp, amp_percentile)

    output_lch[:, :, 0] = (
            lightness_min + (np.clip(amp / maxamp, 0, 1)) * lightness_delta
    )
    output_lch[:, :, 1] = (np.clip(amp / maxamp, 0, 1)) * maxsat
    output_lch[:, :, 2] = (-angle + hueshift) * 180 / np.pi

    return JCh_to_RGB255(output_lch)

In [ ]:
master = Path(r"\\funes2\legacy\experiments\E0040_motions_cardinal\v10_ls_10p10s\2p\huc")
save_master = Path(r"Z:\Hagar\mpin data e0040\v10_2p_huc")

all_fish = list(master.glob("*_f*"))
all_fish_save = list(save_master.glob("*_f*"))

fish_dir = all_fish[2]
save_dir = all_fish_save[1]

print(fish_dir)
print(save_dir)

In [ ]:
source_dir = "anatomy\original"
source_dir = "original"

frames_per_plane = None
aligned_dir = "aligned"
force_realign = False
force_remake_tifs = False
force_remake_corr = False

In [ ]:
if not(fish_dir / 'aligned').exists() or force_realign:
    align_2p_volume(SplitDataset(f / source_dir), output_dir=str(save_dir), across_planes=False)
else:
    print("Already aligned")

In [ ]:
if not(save_dir / 'anatomy.tif').exists() or force_remake_tifs:
    ds = SplitDataset(fish_dir / "aligned")
    stack = ds[:, :, :, :]
    mean = stack.mean(0)
    print(np.shape(mean))
    tiff.imsave(str(save_dir / "anatomy.tif"), mean)
    mean = 0
    stack = 0
else:
    print("Already anatomized")

In [ ]:
#### calculate tuning maps

aligned = SplitDataset(fish_dir / "dff")
stack = aligned[:, :, :, :]

In [ ]:
exp_list = glob(str(fish_dir / "behavior/*.json"))
beh_path = fish_dir / 'behavior'

metadata_file = list(beh_path.glob("*_metadata.json"))[0]

with open(str(metadata_file), "r") as f:
     metadata = json.load(f)
stim = metadata["stimulus"]["log"]

fs = int(metadata['imaging']['microscope_config']['scanning']['framerate'])


sampling = 1/fs
time = np.linspace(0, aligned.shape[0]*sampling, aligned.shape[0])
len_rec, num_planes, x_pix, y_pix = np.shape(aligned)
np.shape(aligned)

In [ ]:
# make a list of sensory regressors for each plane
reg_list = [make_sensory_regressors(Experiment(exp), sampling=sampling) for exp in exp_list]

In [ ]:
print(np.shape(aligned))
print(np.shape(reg_list))

In [ ]:
# calculate tuning

amps = []
angles = []
for i in range(stack.shape[1]):
    print(i)
    img = stack[:,i,:,:]
    amp, angle = get_tuning_map(img, reg_list[i])
    amps.append(amp)
    angles.append(angle)

df = pd.DataFrame(list(zip(amps, angles)), columns=["amp", "angle"])

In [ ]:
# make a color map from the amplitude/angle

pctl = 90

color_maps = []
for i in range(stack.shape[1]):
    amp = df.loc[i, "amp"]
    angle = df.loc[i, "angle"]
    color_map = color_stack(np.nan_to_num(amp), np.nan_to_num(angle), amp_percentile=pctl) #default percentile was 80
    color_maps.append(color_map)
    
color_maps = np.array(color_maps)

In [ ]:
fl.save(save_dir / "tuning_map_90_fixed_2024.h5", color_maps)

In [ ]:
with napari.gui_qt():
    v = napari.view_image(color_maps)

In [ ]:
num_row = 3
num_col = 3
fig, axs = plt.subplots(num_row, num_col, figsize=(10, 10), sharey=True, sharex=True)

count = 0 

for i in range(0, num_planes):
    r = i // num_col
    c = np.mod(i, num_col)
    
    if count > 0:
        axs[r,c].axis('off')
    else:
        count += 1
        
    tmp_plane = np.rot90(color_maps[i], 2)
    #tmp_plane = np.ma.masked_where(tmp_anatomy < 1, tmp_plane)
    axs[r,c].imshow(tmp_plane)
    


In [ ]:
fig.savefig(fish / "tuning_curve2.pdf", dpi=300)
fig.savefig(fish / "tuning_curve2.jpg", dpi=300)